# Recursive Forecasting with future known features

[Forecasting with Machine Learning - Course](https://www.trainindata.com/p/forecasting-with-machine-learning)

In this notebook, we carry out recursive forecasting to predict multiple steps into the future by using a Lasso regression.

SKForecast produces lags out of the box. But we can combine SKForecast with Feature-engine and other libraries to create more features.

In this notebook, we will add features whose values in the future we know: **features about date and time.**

In [27]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

#from skforecast.ForecasterAutoreg import ForecasterAutoreg
from skforecast.recursive import ForecasterRecursive
from feature_engine.datetime import DatetimeFeatures

# Load data

We will use the electricity demand dataset found [here](https://github.com/tidyverts/tsibbledata/tree/master/data-raw/vic_elec/VIC2015).

**Citation:**

Godahewa, Rakshitha, Bergmeir, Christoph, Webb, Geoff, Hyndman, Rob, & Montero-Manso, Pablo. (2021). Australian Electricity Demand Dataset (Version 1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.4659727

**Description of data:**

A description of the data can be found [here](https://rdrr.io/cran/tsibbledata/man/vic_elec.html). The data contains electricity demand in Victoria, Australia, at 30 minute intervals over a period of 12 years, from 2002 to early 2015. There is also the temperature in Melbourne at 30 minute intervals and public holiday dates.

In [ ]:
# Electricity demand.
url = "https://raw.githubusercontent.com/tidyverts/tsibbledata/master/data-raw/vic_elec/VIC2015/demand.csv"
df = pd.read_csv(url)

df.drop(columns=["Industrial"], inplace=True)

# Convert the integer Date to an actual date with datetime type
df["date"] = df["Date"].apply(
    lambda x: pd.Timestamp("1899-12-30") + pd.Timedelta(x, unit="days")
)

# Create a timestamp from the integer Period representing 30 minute intervals
df["date_time"] = df["date"] + \
    pd.to_timedelta((df["Period"] - 1) * 30, unit="m")

df.dropna(inplace=True)

# Rename columns
df = df[["date_time", "OperationalLessIndustrial"]]

df.columns = ["date_time", "demand"]

# Resample to hourly
df = (
    df.set_index("date_time")
    .resample("h")
    .agg({"demand": "sum"})
)

df.head()

In [ ]:
df.tail()

In [30]:
# Split into train and test

# We leave the last February in the test set

end_train = '2014-12-31 23:59:59'
X_train = df.loc[:end_train]
X_test  = df.loc[end_train:]

## Plot time series

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
X_train.tail(500).plot(ax=ax)
X_test.head(500).plot(ax=ax)
ax.set_title('Hourly energy consumption.')
ax.legend(["train", "test"])
plt.show()

## Regression model

In [32]:
# Lasso regression model
# datetime and energy demand are not in the
# same scale, so we need to scale the variables

model = Pipeline([
    ("scaler", MinMaxScaler()),
    ("lasso", Lasso(random_state=9, alpha=10))
])

## Adding datetime features

Let's now do recursive forecasting adding datetime features.

In [33]:
# Feature engine's transformer to create datetime
# features can extract features from one or more columns
# or from the index.

# To be compatible with skforecast, we need to extract
# features from a column, and ensure the result does not contain
# any variables that we do not want as part of the forecast.

datetime_f = DatetimeFeatures(
    features_to_extract = ["month", "day_of_week", "hour"],
    drop_original=True,
)

In [ ]:
# the input to the datetime features

datetime_df = pd.DataFrame(
    X_train.index,
    index=X_train.index,
)

# the index needs to match with the series we
# want to forecast

datetime_df

In [ ]:
# test transformer

datetime_f.fit_transform(datetime_df)

The output of the transformer is the datetime features that we want. That's good.

**Important**: make sure you don accidentally introduce any data that leaks information about the future!!!

Now, we can plug it into the forecaster:

In [36]:
forecaster = ForecasterRecursive(
    regressor=model,              # the machine learning model
    lags=[1, 24, 7*24],           # the lag features to create
    transformer_exog=datetime_f,  # to get the datetime features
    forecaster_id="recursive"
)

In [ ]:
# Fit the model to the data

forecaster.fit(
    y=X_train["demand"],  # the series for the lags
    exog=datetime_df,    # the datetime for the datetime features
)

forecaster

## Input features to regression

In [ ]:
# Check the predictor features table created by skforecast.

# These are the input to the Lasso, so it's important to know
# what we are using for training:

X, y = forecaster.create_train_X_y(
    y=X_train["demand"],
    exog=datetime_df,
)

X, y

That is the data **before** scaling.

`2002-01-07` was a Monday, to double check the `day of the week variable`.

In [ ]:
# If we print the last points in the time series before 
# the starting point in the training set, # we can corroborate
# lag of 1. If we print further back, we can corroborate lag of 24 and so on

# the first points were dropped, because they contained the NAN values
# introduced by lag of 24 and lag of 144. So in short, the first 144 rows
# are dropped.

X_train.loc[:"2002-01-07 00:00:00"].tail()

## Forecast next 24 hs

We will predict the first points of energy demand right after the training set.

That is, starting `2015-01-01 00:00:00`.

In [ ]:
# we need to create the table with datetime from
# which the datetime features will be created

datetime_df_test = pd.DataFrame(
    X_test.head(24).index,
    index=X_test.head(24).index,
)

# this date needs to coincide with the forecasting
# horizon
datetime_df_test.head()

In [ ]:
# Predict the next 24 hs

predictions = forecaster.predict(
    steps=24,
    exog=datetime_df_test,
)

predictions.head()

In [ ]:
# Plot the forecast vs the actual

fig, ax = plt.subplots(figsize=(6, 3))
X_train.tail(100)["demand"].plot(ax=ax, label='train')
X_test.head(24)["demand"].plot(ax=ax, label='test')
predictions.plot(ax=ax, label='predictions')
plt.title("Lasso forecasting")
plt.ylabel("Energy demand (hourly)")
ax.legend(bbox_to_anchor=(1.3, 1.0));

In [ ]:
# Prediction error

error_mse = mean_squared_error(
                y_true = X_test["demand"].head(24),
                y_pred = predictions
            )

print(f"Test error (mse): {error_mse}")

In [ ]:
# Prediction error

error_rmse = root_mean_squared_error(
                y_true = X_test["demand"].head(24),
                y_pred = predictions,
            )

print(f"Test error (rmse): {error_rmse}")

## Predict any time point into the future

In [ ]:
# Say we want to predict energy demand for 1st of February

forecast_start = '2015-02-01 00:00:00'

# we need the energy demand up to 144 hs before that point
past_data_available = X_test[:'2015-01-31 23:59:59'].tail(168)

# data in the past that we know at the point of forecast
past_data_available.head()

In [ ]:
# we also need the datetime in the date range of the forecasting horizon.

# To make predictions `exog` must start one step ahead of `last_window`.

horizon = X_test['2015-01-31 23:59:59':].head(24)

datetime_df_test = pd.DataFrame(
    horizon.index,
    index=horizon.index,
)

datetime_df_test

In [ ]:
predictions = forecaster.predict(
    steps=24,
    last_window=past_data_available["demand"],
    exog=datetime_df_test,
)

predictions.head()

In [ ]:
# Plot the forecast vs the actual

fig, ax = plt.subplots(figsize=(6, 3))
horizon["demand"].plot(ax=ax, label='actuals')
predictions.plot(ax=ax, label='predictions')
plt.title("Lasso forecasting")
plt.ylabel('Energy demand (hourly)')
ax.legend(bbox_to_anchor=(1.3, 1.0));

In [ ]:
# Prediction error

error_mse = mean_squared_error(
                y_true = horizon,
                y_pred = predictions
            )

print(f"Test error (mse): {error_mse}")

In [ ]:
# Prediction error

error_rmse = root_mean_squared_error(
                y_true = horizon,
                y_pred = predictions,
            )

print(f"Test error (rmse): {error_rmse}")

## Feature importance

In [ ]:
pd.Series(forecaster.regressor.named_steps["lasso"].coef_,
          index=forecaster.regressor.feature_names_in_).plot.bar()
plt.title('Feature importance')
plt.ylabel("Coefficients")
plt.show()